# Speeding up a Python loop with Cython

This notebook starts from a slow Python function, the computation of an image of the Mandelbrot set, and speeds it up step by step with Cython:

0. Where does the time go?
1. Compiling the Python code as is
2. Typing the variables with `cdef`
3. `cpdef` and a typed view on the array
4. A package with `pyproject.toml`
5. Going further: pure Python mode

Each step starts from the previous one and times its gain with `%timeit`. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed.

Cython translates the code into C, which `gcc` compiles: both are installed in Colab as in the Codespace. The first cell loads IPython's `cython` extension, which adds the `%%cython` magic command.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%load_ext cython

## The starting point in Python

For each point `c = cx + i·cy` of the plane, the sequence `z ← z² + c`, started from `z = 0`, stays bounded or escapes to infinity. `iterations_py` counts the iterations before `|z|` goes over 2, at most `max_iter`. `mandelbrot_py` makes an image of it: one pixel per point of a rectangle of the plane. The Mandelbrot set is made of the points that reach `max_iter`.

In [ ]:
# The iterations of z = z² + c from z = 0, with z = x + i·y and c = cx + i·cy,
# until |z| goes over 2 (at most max_iter).
def iterations_py(cx, cy, max_iter):
  x = y = 0.0
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_py(width, height, max_iter):
  image = np.zeros((height, width), dtype=np.int32)
  for i in range(height):
    for j in range(width):
      # The point of the plane where pixel (i, j) is.
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      image[i, j] = iterations_py(cx, cy, max_iter)
  return image

In [ ]:
image = mandelbrot_py(480, 320, 100)
plt.imshow(image, cmap="magma")
plt.axis("off")
plt.show()

The reference time, Python's:

In [ ]:
%timeit mandelbrot_py(480, 320, 100)

## 0. Where does the time go?

Before speeding anything up, profile: `cProfile` counts the calls of each function and measures the time spent in each. Profile `mandelbrot_py(480, 320, 100)` with `cProfile.run`, sorting by own time (`sort="tottime"`). How many times is `iterations_py` called, and what share of the total time is spent in it? Which function should be sped up first?

**Useful documentation**

- https://docs.python.org/3/library/profile.html#instant-user-s-manual

In [ ]:
# Your code here

### Additional hints

`cProfile.run` takes the code to profile as a string, and prints one line per function: `ncalls` (number of calls), `tottime` (time spent in the function itself, without the functions it calls), `cumtime` (with them). In Jupyter, `%prun mandelbrot_py(480, 320, 100)` does the same.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import cProfile

cProfile.run(..., sort="tottime")
```

### Solution

In [ ]:
import cProfile

cProfile.run("mandelbrot_py(480, 320, 100)", sort="tottime")
# iterations_py is called 153,600 times, once per pixel (480 × 320), and takes
# almost all the time: it, and the loop that calls it, should be sped up first.

## 1. Compiling the Python code as is

A cell starting with `%%cython` is translated into C by Cython, compiled, then imported into the notebook. The next cell holds `iterations_py` and `mandelbrot_py`, copied without changing their code and renamed `iterations_v1` and `mandelbrot_v1`. Run it, then, in the second cell, time `mandelbrot_v1(480, 320, 100)` with `%timeit`. Is the gain worth it?

In [ ]:
%%cython
import numpy as np


def iterations_v1(cx, cy, max_iter):
  x = y = 0.0
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v1(width, height, max_iter):
  image = np.zeros((height, width), dtype=np.int32)
  for i in range(height):
    for j in range(width):
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      image[i, j] = iterations_v1(cx, cy, max_iter)
  return image

In [ ]:
# Your code here

### Additional hints

The `%%cython` cell is a module of its own: it doesn't see the notebook's variables, so it imports NumPy itself. The functions it defines, though, are imported into the notebook.

`%timeit` followed by an expression runs it several times and prints its mean time. Compare it with `mandelbrot_py`'s reference time.

A skeleton to copy into the exercise's second cell, replacing the `...`:

```python
%timeit mandelbrot_v1(...)
```

### Solution

In [ ]:
%timeit mandelbrot_v1(480, 320, 100)
# No faster than Python, often slower: without types, the generated C
# handles the same Python objects as the interpreter.

## 2. Typing the variables with `cdef`

`%%cython -a` compiles the cell and shows the annotated code: the yellower a line, the more it calls the Python interpreter; clicking a line's `+` shows the generated C. Write `iterations_v2` and `mandelbrot_v2` in a `%%cython -a` cell:

- `iterations_v2` becomes a C function, declared with `cdef int`, with typed arguments (`double cx`, `double cy`, `int max_iter`), as are its local variables (`x`, `y`, `n`);
- `mandelbrot_v2` stays a `def` function, called from Python, with its arguments and its variables (`i`, `j`, `cx`, `cy`) typed.

In the second cell, check with `np.array_equal` that `mandelbrot_v2(480, 320, 100)` gives the same image as `image`, and time it. Which line of the loop stays yellow?

In [ ]:
%%cython -a
# Your code here

In [ ]:
# Your code here

### Additional hints

- Arguments are typed in the signature, `def f(int n)`; local variables with `cdef`, at the top of the function: `cdef double x = 0.0, y = 0.0`, `cdef int i, j`.
- A `cdef` function's return type goes before its name: `cdef int f(...)`.
- The rest of the code doesn't change.

A skeleton to copy into the exercise's first cell, replacing each `...`:

```cython
%%cython -a
import numpy as np


cdef int iterations_v2(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  ...


def mandelbrot_v2(int width, int height, int max_iter):
  cdef int i, j
  cdef double cx, cy
  ...
```

Then, in the second cell:

```python
print(np.array_equal(mandelbrot_v2(480, 320, 100), image))
%timeit mandelbrot_v2(480, 320, 100)
```

### Solution

In [ ]:
%%cython -a
import numpy as np


cdef int iterations_v2(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v2(int width, int height, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((height, width), dtype=np.int32)
  for i in range(height):
    for j in range(width):
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      image[i, j] = iterations_v2(cx, cy, max_iter)
  return image

In [ ]:
print(np.array_equal(mandelbrot_v2(480, 320, 100), image))
%timeit mandelbrot_v2(480, 320, 100)
# Ten to twenty times faster than Python. The line image[i, j] = ... stays
# yellow: indexing a NumPy array goes through Python.

## 3. `cpdef` and a typed view on the array

A `cdef` function only exists on the C side: Python code doesn't see `iterations_v2`.

In [ ]:
try:
  iterations_v2(1.0, 1.0, 100)
except NameError as error:
  print("Error:", error)

Write `iterations_v3` and `mandelbrot_v3` in a new `%%cython -a` cell:

- declare `iterations_v3` with `cpdef`: it stays a fast C function for `mandelbrot_v3`, and can also be called from Python;
- in `mandelbrot_v3`, write into the array through a typed view, `cdef int[:, :] view = image`, rather than through `image[i, j]`.

In the second cell, print `iterations_v3(0.0, 0.0, 100)` and `iterations_v3(1.0, 1.0, 100)`, which are `100` and `2`, check that the image is the same, and time `mandelbrot_v3`. Is there any yellow left in the loop?

**Useful documentation**

- https://cython.readthedocs.io/en/latest/src/userguide/memoryviews.html

In [ ]:
%%cython -a
# Your code here

In [ ]:
# Your code here

### Additional hints

- `cpdef int iterations_v3(...)`: only the keyword changes from `iterations_v2`.
- `int[:, :]` is a view on a two-dimensional array of C `int`s: the `np.int32` array `np.zeros` creates is one. `view[i, j] = ...` then becomes a direct memory access, and `image`, which shares that memory, holds the result.

A skeleton to copy into the exercise's first cell, replacing each `...`:

```cython
%%cython -a
import numpy as np


cpdef int iterations_v3(double cx, double cy, int max_iter):
  ...


def mandelbrot_v3(int width, int height, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((height, width), dtype=np.int32)
  cdef int[:, :] view = ...
  ...
  return image
```

Then, in the second cell:

```python
print(iterations_v3(0.0, 0.0, 100), iterations_v3(1.0, 1.0, 100))
print(np.array_equal(mandelbrot_v3(480, 320, 100), image))
%timeit mandelbrot_v3(480, 320, 100)
```

### Solution

In [ ]:
%%cython -a
import numpy as np


cpdef int iterations_v3(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v3(int width, int height, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((height, width), dtype=np.int32)
  cdef int[:, :] view = image
  for i in range(height):
    for j in range(width):
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      view[i, j] = iterations_v3(cx, cy, max_iter)
  return image

In [ ]:
print(iterations_v3(0.0, 0.0, 100), iterations_v3(1.0, 1.0, 100))
print(np.array_equal(mandelbrot_v3(480, 320, 100), image))
%timeit mandelbrot_v3(480, 320, 100)
# Two to three times faster again. A very pale yellow remains in the loop: a
# division by zero check (cx, cy), an index check (view[i, j]) and an exception
# check after the call to iterations_v3.

## 4. A package with `pyproject.toml`

Outside a notebook, Cython code lives in a `.pyx` file, and a package distributes it, compiled at install time. The next two cells create the `mandel` package in the `mandel_package` folder: version 3 in `src/mandel/core.pyx`, and a `src/mandel/__init__.py` that imports `mandelbrot` from it. Write the two files that build it:

- `mandel_package/pyproject.toml`: the setuptools *backend*, with Cython among the build dependencies, then the package's name (`mandel`) and version;
- `mandel_package/setup.py`: a minimal `setup()`, which declares the extension with `cythonize`.

In the next cell, install the package with `%pip install ./mandel_package`, import `mandel`, check that `mandel.mandelbrot(480, 320, 100)` gives the same image as `image` and time it.

**Useful documentation**

- https://cython.readthedocs.io/en/latest/src/userguide/source_files_and_compilation.html#basic-setup-py
- https://packaging.python.org/en/latest/guides/writing-pyproject-toml/

In [ ]:
from pathlib import Path

package = Path("mandel_package/src/mandel")
package.mkdir(parents=True, exist_ok=True)
with (package / "__init__.py").open("w") as f:
  f.write("from .core import mandelbrot\n")

In [ ]:
%%writefile mandel_package/src/mandel/core.pyx
# cython: language_level=3
import numpy as np


cpdef int iterations(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot(int width, int height, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((height, width), dtype=np.int32)
  cdef int[:, :] view = image
  for i in range(height):
    for j in range(width):
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      view[i, j] = iterations(cx, cy, max_iter)
  return image

In [ ]:
%%writefile mandel_package/pyproject.toml
# Your code here

In [ ]:
%%writefile mandel_package/setup.py
# Your code here

In [ ]:
# Your code here

### Additional hints

- `pyproject.toml` has two sections: `[build-system]`, with `requires` (the packages installed for the build: `setuptools` and `cython`) and `build-backend = "setuptools.build_meta"`; `[project]`, with `name` and `version`.
- `setup.py` only calls `setup(ext_modules=cythonize(...))`, with the `.pyx` path relative to `mandel_package`; setuptools finds the `mandel` package under `src/` by itself.
- `pip install` builds the package in an environment of its own, installs Cython there, compiles the extension, then installs the result: allow a minute. Python imports a module only once: after changing the package, reinstall it and restart the session before importing it again.

A skeleton to copy into the exercise's three cells, replacing each `...`:

```toml
%%writefile mandel_package/pyproject.toml
[build-system]
requires = [...]
build-backend = ...

[project]
name = "mandel"
version = "0.1.0"
```

```python
%%writefile mandel_package/setup.py
from Cython.Build import cythonize
from setuptools import setup

setup(ext_modules=cythonize(...))
```

```python
%pip install ./mandel_package
import mandel

print(np.array_equal(mandel.mandelbrot(480, 320, 100), image))
%timeit mandel.mandelbrot(480, 320, 100)
```

### Solution

In [ ]:
%%writefile mandel_package/pyproject.toml
[build-system]
requires = ["setuptools", "cython>=3"]
build-backend = "setuptools.build_meta"

[project]
name = "mandel"
version = "0.1.0"

In [ ]:
%%writefile mandel_package/setup.py
from Cython.Build import cythonize
from setuptools import setup

setup(ext_modules=cythonize("src/mandel/core.pyx"))

In [ ]:
%pip install ./mandel_package
import mandel

print(np.array_equal(mandel.mandelbrot(480, 320, 100), image))
%timeit mandel.mandelbrot(480, 320, 100)
# As fast as mandelbrot_v3: the same code, compiled at install time.

## 5. Going further: pure Python mode

Cython 3 completes pure Python mode: an ordinary `.py` file carries its types as annotations: `x: cython.double`, `@cython.cfunc` for a `cdef` function, `@cython.ccall` for a `cpdef` function, `cython.int[:, :]` for a typed view. The file stays valid Python, which runs without compilation, and `cythonize -i` compiles it as it does a `.pyx`. In the first cell, write version 3 this way into `mandel_pure.py`. In the second, compile it, import `mandel_pure`, print `mandel_pure.__file__` to see which file Python loaded, and time it.

**Useful documentation**

- https://cython.readthedocs.io/en/latest/src/tutorial/pure.html

In [ ]:
%%writefile mandel_pure.py
# Your code here

In [ ]:
# Your code here

### Additional hints

- Each typed variable is declared by an annotation, one per line: `i: cython.int`.
- A function's return type is its return annotation: `-> cython.int`.
- When `mandel_pure.py` and the compiled module sit side by side, `import mandel_pure` loads the compiled module: `__file__` ends with `.so`.

A skeleton to copy into the exercise's two cells, replacing each `...`:

```python
%%writefile mandel_pure.py
# cython: language_level=3
import cython
import numpy as np


@cython.ccall
def iterations(
  cx: cython.double, cy: cython.double, max_iter: cython.int
) -> cython.int:
  x: cython.double = 0.0
  ...


def mandelbrot(width: cython.int, height: cython.int, max_iter: cython.int):
  i: cython.int
  ...
  view: cython.int[:, :] = image
  ...
```

Then, in the second cell:

```python
!cythonize -i mandel_pure.py
import mandel_pure

print(mandel_pure.__file__)
print(np.array_equal(mandel_pure.mandelbrot(480, 320, 100), image))
%timeit mandel_pure.mandelbrot(480, 320, 100)
```

### Solution

In [ ]:
%%writefile mandel_pure.py
# cython: language_level=3
import cython
import numpy as np


@cython.ccall
def iterations(
  cx: cython.double, cy: cython.double, max_iter: cython.int
) -> cython.int:
  x: cython.double = 0.0
  y: cython.double = 0.0
  n: cython.int
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot(width: cython.int, height: cython.int, max_iter: cython.int):
  i: cython.int
  j: cython.int
  cx: cython.double
  cy: cython.double
  image = np.zeros((height, width), dtype=np.int32)
  view: cython.int[:, :] = image
  for i in range(height):
    for j in range(width):
      cx = -2.0 + 3.0 * j / width
      cy = -1.2 + 2.4 * i / height
      view[i, j] = iterations(cx, cy, max_iter)
  return image

In [ ]:
!cythonize -i mandel_pure.py
import mandel_pure

print(mandel_pure.__file__)
print(np.array_equal(mandel_pure.mandelbrot(480, 320, 100), image))
%timeit mandel_pure.mandelbrot(480, 320, 100)
# The imported module is the compiled .so, as fast as mandel.